# Advanced 10 — Identity Observability, Telemetry & Forensics for Agents

**Scenario:** investigate a high-risk claims-agent execution across a human session, orchestrator, research sub-agent, workload identity, authorization service, MCP tool, downstream API and credential broker.

The objective is to reconstruct **identity + authority + execution**, not simply application logs.

All examples are local simulations.


In [ ]:
from datetime import datetime,timedelta,timezone
import json, uuid, hashlib, hmac, re, statistics
import pandas as pd, networkx as nx
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor, ConsoleSpanExporter
trace.set_tracer_provider(TracerProvider())
tracer=trace.get_tracer("agent-identity-course")
NOW=datetime.now(timezone.utc)


## Lab 1 — Normalized identity event

In [ ]:
event={"schema_name":"agent.identity.event","schema_version":"1.0",
"event_type":"authorization.decision","timestamp":NOW.isoformat(),"trace_id":uuid.uuid4().hex,
"actor":{"id":"agent:research","type":"agent"},"subject":{"id":"user:alice"},
"action":"claims.read","resource":"claim:12345","decision":"permit",
"policy":{"id":"claims-policy","version":"17"},"delegation_id":"dlg-882"}
event

## Lab 2 — Create OpenTelemetry spans

In [ ]:
with tracer.start_as_current_span("invoke_agent") as root:
    root.set_attribute("agent.id","claims-agent")
    with tracer.start_as_current_span("authorization") as s:
        s.set_attribute("authz.decision","permit")
        s.set_attribute("authz.policy.version","17")
    with tracer.start_as_current_span("execute_tool") as s:
        s.set_attribute("tool.name","policy-search")

## Lab 3 — Trace identifiers

In [ ]:
with tracer.start_as_current_span("example") as s:
    ctx=s.get_span_context()
    ids={"trace_id":format(ctx.trace_id,"032x"),"span_id":format(ctx.span_id,"016x")}
ids

## Lab 4 — Separate trace and identity context

In [ ]:
context={"trace":{"trace_id":"abc","span_id":"def"},
"identity":{"actor":"agent:research","subject":"user:alice","workload":"spiffe://prod/agents/research"}}
context

## Lab 5 — Agent trace tree

In [ ]:
trace_tree=[("invoke_agent",None),("model_call","invoke_agent"),("authorize","invoke_agent"),
("execute_tool","invoke_agent"),("downstream_api","execute_tool")]
pd.DataFrame(trace_tree,columns=["span","parent"])

## Lab 6 — Authorization decision event

In [ ]:
authz={"request_id":"authz-44","principal":"agent:research","subject":"user:alice",
"action":"claims.read","resource":"claim:12345","decision":"permit","policy_id":"claims","policy_version":"17"}
authz

## Lab 7 — PDP to PEP correlation

In [ ]:
pdp={"request_id":"authz-44","decision":"permit"}
pep={"request_id":"authz-44","enforced":True,"result":"request forwarded"}
pdp["request_id"]==pep["request_id"] and pep["enforced"]

## Lab 8 — Detect permit without enforcement evidence

In [ ]:
decisions={"a1":"permit","a2":"deny","a3":"permit"}
enforced={"a1":True,"a2":True}
[x for x,d in decisions.items() if d=="permit" and x not in enforced]

## Lab 9 — Credential lifecycle

In [ ]:
credential_events=pd.DataFrame([
{"t":1,"credential":"fp:a9","event":"issued"},{"t":2,"credential":"fp:a9","event":"exchanged"},
{"t":5,"credential":"fp:a9","event":"revoked"}])
credential_events

## Lab 10 — Token exchange event

In [ ]:
exchange={"client":"agent:research","subject":"user:alice","actor":"agent:research",
"input_credential":"fp:a9","audience":"claims-api","scope":["claims.read"],"output_credential":"fp:b7",
"trace_id":"trace-42"}
exchange

## Lab 11 — Delegation event

In [ ]:
delegation={"id":"dlg-9","delegator":"agent:claims","delegate":"agent:research",
"subject":"user:alice","actions":["documents.read"],"resources":["vector:index:claims"],
"depth":1,"redelegation":False,"expires_at":(NOW+timedelta(minutes=20)).isoformat(),"trace_id":"trace-42"}
delegation

## Lab 12 — Delegation lineage graph

In [ ]:
DG=nx.DiGraph()
DG.add_edge("user:alice","agent:claims",relation="sponsors")
DG.add_edge("agent:claims","agent:research",relation="delegates",delegation="dlg-9")
DG.add_edge("agent:research","agent:data",relation="delegates",delegation="dlg-10")
nx.shortest_path(DG,"user:alice","agent:data")

## Lab 13 — Tool telemetry

In [ ]:
tool={"trace_id":"trace-42","actor":"agent:research","subject":"user:alice",
"tool":"policy-search","action":"search","target":"vector:index:claims","authz_request":"authz-44",
"delegation":"dlg-9","result":"success"}
tool

## Lab 14 — MCP boundary identities

In [ ]:
mcp_chain=["agent:research","mcp-client:runtime-7","mcp-server:policy","tool:search","api:documents"]
list(zip(mcp_chain,mcp_chain[1:]))

## Lab 15 — Cloud workload event

In [ ]:
cloud={"principal":"arn:aws:sts::123:assumed-role/ClaimsAgent/session",
"workload":"spiffe://prod/claims/agent","action":"kms:Sign","resource":"kms:key:claims","trace_id":"trace-42"}
cloud

## Lab 16 — Federation event

In [ ]:
federation={"foreign_trust_domain":"partner.example","subject":"spiffe://partner.example/agent/r1",
"bundle_version":"sha256:123","validation":"success","audience":"research-gateway","trace_id":"trace-42"}
federation

## Lab 17 — CAEP-style security signal

In [ ]:
signal={"type":"risk-level-change","subject":"agent:research","previous":"low","current":"high",
"event_time":NOW.isoformat(),"recommended_action":"reevaluate"}
signal

## Lab 18 — Schema versioning

In [ ]:
schemas={"agent.identity.event":["1.0","1.1"],"agent.delegation.event":["1.0"]}
schemas

## Lab 19 — Cross-system correlation

In [ ]:
events=[
{"system":"agent","trace":"t1","delegation":"d1"},{"system":"pdp","trace":"t1","delegation":"d1"},
{"system":"tool","trace":"t1","delegation":"d1"},{"system":"cloud","trace":"t1","delegation":"d1"}]
pd.DataFrame(events).groupby(["trace","delegation"])["system"].apply(list)

## Lab 20 — Occurrence vs ingestion time

In [ ]:
e={"occurred":NOW,"ingested":NOW+timedelta(seconds=7)}
(e["ingested"]-e["occurred"]).total_seconds()

## Lab 21 — Causal reconstruction

In [ ]:
causal=nx.DiGraph()
causal.add_edges_from([("user_request","agent_invocation"),("agent_invocation","delegation"),
("delegation","authorization"),("authorization","tool_call"),("tool_call","api_read")])
list(nx.topological_sort(causal))

## Lab 22 — Governance enrichment

In [ ]:
raw={"actor":"agent:research"}
inventory={"agent:research":{"owner":"claims-ai","risk_tier":"high","environment":"prod","purpose":"policy research"}}
{**raw,**inventory[raw["actor"]]}

## Lab 23 — Secret redaction

In [ ]:
SENSITIVE={"access_token","refresh_token","api_key","client_secret","authorization"}
def redact(d): return {k:("[REDACTED]" if k.lower() in SENSITIVE else v) for k,v in d.items()}
redact({"agent":"a1","access_token":"ey-secret","scope":"read"})

## Lab 24 — Credential fingerprint

In [ ]:
def fingerprint(identifier,key=b"demo-key"):
    return hmac.new(key,identifier.encode(),hashlib.sha256).hexdigest()
fingerprint("credential-instance-44")[:16]

## Lab 25 — Security-aware sampling

In [ ]:
traces=[
{"id":"t1","risk":10,"denied":False,"kms":False},{"id":"t2","risk":90,"denied":True,"kms":False},
{"id":"t3","risk":40,"denied":False,"kms":True}]
[x["id"] for x in traces if x["risk"]>=70 or x["denied"] or x["kms"]]

## Lab 26 — Tail sampling

In [ ]:
def retain(t):
    return t.get("error") or t.get("denied") or t.get("risk",0)>=70 or t.get("critical_tool")
retain({"risk":10,"critical_tool":True})

## Lab 27 — Hash-chain evidence

In [ ]:
def canon(x):return json.dumps(x,sort_keys=True,separators=(",",":"))
def chain(events):
    out=[];prev=""
    for e in events:
        h=hashlib.sha256((prev+canon(e)).encode()).hexdigest()
        out.append({"event":e,"prev":prev,"hash":h});prev=h
    return out
evidence_chain=chain([{"e":"issued"},{"e":"delegated"},{"e":"accessed"}])
evidence_chain

## Lab 28 — Verify hash chain

In [ ]:
def verify(c):
    prev=""
    for x in c:
        if x["prev"]!=prev:return False
        if hashlib.sha256((prev+canon(x["event"])).encode()).hexdigest()!=x["hash"]:return False
        prev=x["hash"]
    return True
verify(evidence_chain)

## Lab 29 — Detect tampering

In [ ]:
tampered=json.loads(json.dumps(evidence_chain))
tampered[1]["event"]["e"]="escalated"
verify(tampered)

## Lab 30 — Merkle root

In [ ]:
def merkle_root(items):
    hs=[hashlib.sha256(canon(x).encode()).digest() for x in items]
    if not hs:return None
    while len(hs)>1:
        if len(hs)%2:hs.append(hs[-1])
        hs=[hashlib.sha256(hs[i]+hs[i+1]).digest() for i in range(0,len(hs),2)]
    return hs[0].hex()
merkle_root([{"e":1},{"e":2},{"e":3}])

## Lab 31 — Evidence checkpoint

In [ ]:
checkpoint={"window":"2026-08-19T10:00Z/10:05Z",
"merkle_root":merkle_root([{"e":1},{"e":2},{"e":3}]),"signer":"kms:evidence-key","status":"demo"}
checkpoint

## Lab 32 — Chain of custody

In [ ]:
custody=pd.DataFrame([
{"step":1,"actor":"collector","action":"collected","hash":"h1"},
{"step":2,"actor":"evidence-service","action":"archived","hash":"h1"},
{"step":3,"actor":"investigator","action":"exported-copy","hash":"h1"}])
custody

## Lab 33 — Forensic timeline

In [ ]:
timeline=pd.DataFrame([
{"t":"10:00:01","event":"agent.invoke","actor":"user:alice"},
{"t":"10:00:04","event":"delegation.issued","actor":"agent:claims"},
{"t":"10:00:05","event":"token.exchanged","actor":"agent:research"},
{"t":"10:00:06","event":"authz.permit","actor":"agent:research"},
{"t":"10:00:07","event":"tool.execute","actor":"agent:research"}])
timeline

## Lab 34 — Effective authority at time T

In [ ]:
authority={"identity_status":"active","credential_status":"active","policy_version":"17",
"delegation":{"scope":{"documents.read"},"expires":"10:20"},"requested":"documents.read"}
authority["requested"] in authority["delegation"]["scope"]

## Lab 35 — Historical policy version

In [ ]:
policies={"16":{"documents.read":"deny"},"17":{"documents.read":"permit"},"18":{"documents.read":"deny"}}
policies["17"]["documents.read"]

## Lab 36 — Detection rules

In [ ]:
def detect(e):
    f=[]
    if e.get("identity_status")=="revoked":f.append("revoked_identity_used")
    if e.get("unexpected_audience"):f.append("audience_anomaly")
    if e.get("telemetry_missing"):f.append("telemetry_gap")
    if e.get("permit") and not e.get("enforced"):f.append("permit_without_enforcement")
    return f
detect({"permit":True,"enforced":False})

## Lab 37 — Missing telemetry

In [ ]:
expected={"agent","pdp","tool","cloud"}
observed={"agent","pdp","cloud"}
expected-observed

## Lab 38 — SIEM enrichment

In [ ]:
alert={"rule":"delegation_escalation","actor":"agent:research"}
context={"owner":"claims-ai","risk":"high","environment":"prod","playbook":"quarantine-and-review","trace":"trace-42"}
{**alert,**context}

## Lab 39 — Case bundle

In [ ]:
case={"case_id":"IR-2026-044","summary":"research agent authority anomaly",
"identities":["user:alice","agent:claims","agent:research"],"delegations":["dlg-9"],
"trace_ids":["trace-42"],"policy_versions":["17"],"evidence_root":checkpoint["merkle_root"]}
case

## Lab 40 — Audit query

In [ ]:
records=pd.DataFrame([
{"critical":True,"authz":True,"enforced":True},{"critical":True,"authz":True,"enforced":False},
{"critical":False,"authz":False,"enforced":True}])
records[(records.critical)&(~records.enforced)]

## Lab 41 — Observability coverage

In [ ]:
coverage={"identities":.98,"authz_correlation":.94,"delegations":.91,
"credential_lifecycle":.88,"tool_identity":.96,"cloud_attribution":.90}
round(100*sum(coverage.values())/len(coverage),1)

## Lab 42 — Trace completeness

In [ ]:
expected={"agent","identity","authorization","tool","downstream","result"}
observed={"agent","identity","authorization","tool","result"}
{"score":len(observed&expected)/len(expected),"missing":expected-observed}

## Lab 43 — Telemetry quality

In [ ]:
quality_events=[{"actor":"a","policy_version":"17"},{"actor":None,"policy_version":"17"},
{"actor":"b","policy_version":None}]
{"missing_actor":sum(e["actor"] is None for e in quality_events),
"missing_policy_version":sum(e["policy_version"] is None for e in quality_events)}

## Lab 44 — Retention routing

In [ ]:
def route(e):
    if e.get("audit_critical"):return "evidence-store"
    if e.get("security"):return "siem"
    if e.get("trace"):return "trace-backend"
    return "metrics/analytics"
[route(x) for x in [{"audit_critical":True},{"security":True},{"trace":True},{}]]

## Lab 45 — Adversarial evidence tampering

In [ ]:
attack=json.loads(json.dumps(evidence_chain))
attack.pop(1)
{"valid_after_deletion":verify(attack),"expected":False}

# Lab 44 — Capstone: reconstruct an agent identity incident

A production claims assistant receives a user request. It delegates research to a sub-agent. The research agent obtains a scoped credential, calls an MCP search tool, then unexpectedly attempts to exchange its credential for `claims.write`.

Your job is to reconstruct:

```text
WHO initiated the task?
WHAT logical agent handled it?
WHICH workload identity executed?
WHICH credential was issued?
WHO delegated to whom?
WHAT authority was delegated?
WHICH policy version permitted/denied each action?
WHICH tool was invoked?
WHICH downstream service was affected?
WHAT security signal fired?
WHEN was the identity quarantined?
WHEN did revocation become effective?
```

Required engineering outcomes:

1. build one correlated trace;
2. create normalized identity/security events;
3. include actor and represented subject separately;
4. reconstruct delegation lineage;
5. reconstruct credential lineage;
6. join authorization decision to enforcement;
7. correlate MCP/tool and downstream API activity;
8. preserve the policy version;
9. detect the scope-escalation attempt;
10. produce a SIEM-ready alert;
11. quarantine/revoke the simulated identity;
12. generate a forensic timeline;
13. create a tamper-evident evidence chain;
14. verify the evidence chain;
15. generate an investigation case bundle;
16. calculate trace and observability coverage;
17. identify missing evidence;
18. recommend instrumentation improvements.

The capstone is successful only when another investigator can reconstruct the incident **without relying on application source code or developer memory**.


# Review questions

1. Why is logging alone insufficient for agent identity?
2. Which identity layers should be kept distinct?
3. How do trace context and identity context differ?
4. Why should authorization decisions identify policy versions?
5. Why must PDP decisions be correlated with PEP enforcement?
6. What should be recorded for token exchange?
7. Why is delegation a first-class event?
8. How should MCP/tool identity be modeled?
9. When can prompt/content telemetry become dangerous?
10. Why should raw bearer credentials never be logged?
11. What is the purpose of credential fingerprinting?
12. Why can ordinary sampling undermine forensic readiness?
13. How can tail sampling help?
14. What does a hash chain prove—and what does it not prove?
15. Why should evidence custody be separate from the agent runtime?
16. How do occurrence and ingestion timestamps differ?
17. Why is timestamp ordering insufficient to establish causality?
18. What should trigger a missing-telemetry alert?
19. Which events belong in a SIEM versus an APM backend?
20. What does it mean to reconstruct effective authority at time T?
